In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error

In [2]:
df = pd.read_csv("uber.csv")
df.head()

,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,52:06.0,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,04:56.0,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,45:00.0,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,22:21.0,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,47:00.0,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [3]:
print("Shape of dataset:",df.shape)
print("\nMissing values:")
print(df.isnull().sum())

Shape of dataset: (200000, 9)

Missing values:
Unnamed: 0           0
key                  0
fare_amount          0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    1
dropoff_latitude     1
passenger_count      0
dtype: int64


In [4]:
df = df.dropna()
print("Shape after removing missing values:",df.shape)

Shape after removing missing values: (199999, 9)


In [5]:
df = df.drop(columns=["Unnamed: 0","key","pickup_datetime"])
df.head()

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,7.5,-73.999817,40.738354,-73.999512,40.723217,1
1,7.7,-73.994355,40.728225,-73.994710,40.750325,1
2,12.9,-74.005043,40.740770,-73.962565,40.772647,1
3,5.3,-73.976124,40.790844,-73.965316,40.803349,3
4,16.0,-73.925023,40.744085,-73.973082,40.761247,5


In [6]:
df =df[(df["fare_amount"]>0) & (df["fare_amount"] < 500)]
print(df.shape)

(199977, 6)


In [7]:
Q1 = df["fare_amount"].quantile(0.25)
Q3 = df["fare_amount"].quantile(0.75)
IQR = Q3 - Q1
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR
df=df[(df["fare_amount"] >= lower_limit) & (df["fare_amount"] <= upper_limit)]
print("Shape after removing outliers: ",df.shape)

Shape after removing outliers:  (182822, 6)


In [8]:
def haversine(lat1,long1,lat2,long2):
    R = 6371
    lat1 = np.radians(lat1)
    long1 = np.radians(long1)
    lat2 = np.radians(lat2)
    long2 = np.radians(long2)
    dlat = lat2 - lat1
    dlong = long2 - long1
    a = np.sin(dlat / 2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlong/2)**2
    distance = 2 * R * np.arcsin(np.sqrt(a))
    return distance

In [9]:
df["distance"] = haversine(df["pickup_latitude"],df["pickup_longitude"],df["dropoff_latitude"],df["dropoff_longitude"])
df.head()

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,distance
0,7.5,-73.999817,40.738354,-73.999512,40.723217,1,1.683323
1,7.7,-73.994355,40.728225,-73.994710,40.750325,1,2.457590
2,12.9,-74.005043,40.740770,-73.962565,40.772647,1,5.036377
3,5.3,-73.976124,40.790844,-73.965316,40.803349,3,1.661683
4,16.0,-73.925023,40.744085,-73.973082,40.761247,5,4.475450


In [10]:
df = df[(df["distance"]>0) & (df["distance"]<100)]
print(df.shape)

(177307, 7)


In [11]:
print(df.corr(numeric_only = True)["fare_amount"].sort_values(ascending = False))

fare_amount          1.000000
distance             0.723423
passenger_count      0.013431
dropoff_longitude    0.002059
pickup_longitude     0.001216
pickup_latitude     -0.002003
dropoff_latitude    -0.002227
Name: fare_amount, dtype: float64


In [12]:
X = df[["pickup_longitude","pickup_latitude","dropoff_longitude","dropoff_latitude","distance","passenger_count"]]
y = df["fare_amount"]
X.head()

,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,distance,passenger_count
0,-73.999817,40.738354,-73.999512,40.723217,1.683323,1
1,-73.994355,40.728225,-73.994710,40.750325,2.457590,1
2,-74.005043,40.740770,-73.962565,40.772647,5.036377,1
3,-73.976124,40.790844,-73.965316,40.803349,1.661683,3
4,-73.925023,40.744085,-73.973082,40.761247,4.475450,5


In [13]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)
print("Training data:",X_train.shape)
print("Testing data:",X_test.shape)

Training data: (141845, 6)
Testing data: (35462, 6)


In [14]:
lr = LinearRegression()
lr.fit(X_train,y_train)
lr_pred = lr.predict(X_test)

In [15]:
rf = RandomForestRegressor(n_estimators = 100,random_state = 42)
rf.fit(X_train,y_train)
rf_pred = rf.predict(X_test)

In [16]:
print("Linear Regression: ")
print("R2: ",r2_score(y_test,lr_pred))
print("RMSE: ",np.sqrt(mean_squared_error(y_test,lr_pred)))
print("MAE: ",mean_absolute_error(y_test,lr_pred))

Linear Regression: 
R2:  0.5475457776173963
RMSE:  2.790980373384424
MAE:  1.9035838795321631


In [17]:
print("\nRandom Forest Regression")

print("R2:", r2_score(y_test, rf_pred))

print("RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))

print("MAE:", mean_absolute_error(y_test, rf_pred))


Random Forest Regression
R2: 0.7274164089557437
RMSE: 2.166303563270809
MAE: 1.5282742823303817
